In [1]:
# ================= Cell 1: Base Configuration and Core Functions =================

# --- Modeling Logic Description ---
# This project utilizes the L-Space modeling methodology derived from Complex Network Theory.
# Definition: Each metro station is abstracted as a "node" (vertex), and the physical tracks 
# directly connecting these stations are abstracted as "edges" (links).
# Rationale: This approach provides the most intuitive representation of the physical 
# topology of the metro system. It serves as the foundation for analyzing network 
# connectivity, physical path lengths, and evaluating system performance under 
# random failures or targeted attacks (Robustness Analysis).

%matplotlib inline

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import os
import glob
from matplotlib import cm
import numpy as np
import warnings
import textwrap

# Suppress non-critical warnings to maintain a clean experimental environment 
# and focus on analytical results.
warnings.filterwarnings('ignore')

# -----------------------------------------------------------
# 1. Environment Configuration
# -----------------------------------------------------------
# Data storage path
DATA_PATH = r'数据爬取\高德API\整合数据（含环和分叉）'

# Font configuration: Ensuring Chinese characters in charts are rendered correctly.
# Note: SimHei is compatible with most Windows environments.
plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['axes.unicode_minus'] = False

# -----------------------------------------------------------
# 2. Data Preprocessing Functions
# -----------------------------------------------------------

def get_city_list(path):
    """
    [Function]: Traverses the data folder to automatically extract all processed city names.
    [Design Advantage]: Enhances scalability; new city data added to the folder will 
    be recognized automatically without modifying the source code.
    """
    if not os.path.exists(path):
        print(f"Path does not exist: {path}")
        return []
    
    files = glob.glob(os.path.join(path, "*_All_Stations.csv"))
    cities = []
    for f in files:
        filename = os.path.basename(f)
        city_name = filename.split('_All_Stations')[0]
        cities.append(city_name)
    return sorted(list(set(cities)))

def load_data(city_name):
    """
    [Function]: Reads data from raw CSV files, performs data cleaning, 
    and constructs a NetworkX topological graph model.
    [Key Procedures]:
    1. Self-loop Filtering: Automatically removes erroneous segments where the 
       Source ID equals the Destination ID to prevent logical redundancy.
    2. Spatial Association: Assigns latitude and longitude coordinates as node attributes 
       to support subsequent geospatial topological plotting.
    """
    station_file = os.path.join(DATA_PATH, f"{city_name}_All_Stations.csv")
    edge_file = os.path.join(DATA_PATH, f"{city_name}_All_Edges.csv")
    
    if not os.path.exists(station_file) or not os.path.exists(edge_file):
        print(f"Data file missing: Unable to load station or edge data for {city_name}")
        return None, None

    try:
        stations_df = pd.read_csv(station_file, encoding='utf-8-sig')
        edges_df = pd.read_csv(edge_file, encoding='utf-8-sig')
        
        # [Data Cleaning] Remove self-loops (Start ID equals End ID)
        edges_df = edges_df[edges_df['from_id'] != edges_df['to_id']] 
        
        if stations_df.empty or edges_df.empty:
            return None, None

        # Initialize undirected graph model
        G = nx.Graph()
        
        # Map Station ID to physical location (coordinates) and station name
        # Using groupby and mean() to eliminate minor coordinate perturbations across records
        nodes_info = stations_df.groupby('station ID').agg({
            '站名': 'first', '经度': 'mean', '纬度': 'mean'
        }).to_dict('index')

        # Inject station nodes and geospatial attributes into the graph
        for s_id, info in nodes_info.items():
            G.add_node(s_id, pos=(info['经度'], info['纬度']), label=info['站名'])

        # Inject track edge attributes and associate them with metro line names
        for _, row in edges_df.iterrows():
            if row['from_id'] in G and row['to_id'] in G:
                G.add_edge(row['from_id'], row['to_id'], line=row['线路'])
        
        return G, edges_df

    except Exception as e:
        print(f"Error occurred while reading data for city {city_name}: {e}")
        return None, None

# -----------------------------------------------------------
# 3. Core Analytical and Visualization Functions
# -----------------------------------------------------------

def visualize_city(city_name):
    """
    [Function]: Computes key metrics for the selected city's metro network, 
    outputs an analysis report, and generates a geospatial topological visualization.
    """
    G, edges_df = load_data(city_name)
    if G is None: return

    # --- A. Metrics Computation and Terminology ---
    num_nodes = G.number_of_nodes() # Total stations
    num_edges = G.number_of_edges() # Total track segments
    
    # 1. Average Degree: Describes the average connectivity of stations.
    # [Interpretation]: Represents the average number of directions per station. 
    # Higher values indicate superior network-wide transfer convenience.
    degrees = dict(G.degree())
    avg_degree = sum(degrees.values()) / num_nodes
    
    # 2. Network Density: Measures the compactness of the metro grid.
    # [Interpretation]: Reflects the degree to which the metro system has formed a "network."
    # Higher values signify denser connections between stations.
    density = nx.density(G)
    
    # 3. Degree Centrality: Identifies core transfer hubs.
    # [Physical Significance]: Major multi-line transfer hubs yield high scores; 
    # they are critical nodes for maintaining operational efficiency.
    degree_cent = nx.degree_centrality(G)
    top_degree = sorted(degree_cent.items(), key=lambda x: x[1], reverse=True)[:5]
    node_labels = nx.get_node_attributes(G, 'label')

    # --- B. Output Text Report ---
    print("1. Average Degree: Describes the average connectivity capacity of stations.")
    print("[Technical Definition]: Measures the fundamental physical connectivity of the network.")
    print("[Physical Significance]: Represents the average directions extending from each station.Higher values ")
    print("            denote better transfer accessibility, reflecting transfer redundancy and comprehensive connectivity.")
    print("2. Network Density: Measures the compactness of the network.")
    print("[Technical Definition]: Quantifies topological compactness and resource allocation efficiency.")
    print("[Physical Significance]: Reflects the maturity of the network structure. Higher values indicate tighter station connections.")
    print("            In rail transit, density is often used to evaluate the evolution from 'linear-radial' to 'mature-grid' structures, ")
    print("            serving as a core macroscopic metric for coverage depth and topological robustness.")
    print("3. Degree Centrality: Identifies core transfer hubs.")
    print("[Technical Definition]: Identifies physical topological hubs within the network.")
    print("[Physical Significance]: Degree Centrality quantifies a station's capability as a convergence point. ")
    print("            Major transfer hubs score highest, representing critical nodes for overall efficiency.")
    print(f"\n--- {city_name} Metro Network Topological Metrics Analysis Report ---")
    print(f"[Network Scale] Total Stations: {num_nodes} | Total Track Segments: {num_edges}")
    print(f"[Connectivity Level] Average Degree: {avg_degree:.2f} (Reflects average transfer options)")
    print(f"[Network Density] {density:.6f} (Higher values indicate a more compact network)")
    print(f"[Top 5 Critical Hub Nodes] (Stations with highest transfer weights):")
    for i, (node_id, val) in enumerate(top_degree):
        name = node_labels.get(node_id, str(node_id))
        print(f"    Rank {i+1}: {name} (Centrality Score: {val:.4f})")

    # --- C. Plotting: Geospatial Reconstruction ---
    plt.figure(figsize=(12, 10))
    ax = plt.gca()
    pos = nx.get_node_attributes(G, 'pos')
    
    # Extract unique lines and map to colors. Using 'tab20' for distinct line differentiation.
    unique_lines = edges_df['线路'].unique()
    colors = cm.get_cmap('tab20', len(unique_lines))
    line_color_map = {line: colors(i % 20) for i, line in enumerate(unique_lines)}

    # 1. Render tracks (edges) by line
    for line in unique_lines:
        line_data = edges_df[edges_df['线路'] == line]
        edge_list = [(u, v) for u, v in zip(line_data['from_id'], line_data['to_id']) if G.has_edge(u, v)]
        if edge_list:
            nx.draw_networkx_edges(G, pos, edgelist=edge_list, 
                                 edge_color=[line_color_map[line]], width=3, alpha=0.8, ax=ax)

    # 2. Render stations (nodes)
    nx.draw_networkx_nodes(G, pos, node_size=20, node_color='black', alpha=0.5, ax=ax)
    
    # 3. Dynamic Legend: Placed outside the plotting area to avoid obscuring the topology.
    legend_elements = [plt.Line2D([0], [0], color=line_color_map[l], lw=4, label=l) for l in unique_lines]
    ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.01, 1), 
              title="Line Legend", fontsize=9, frameon=False)

    plt.title(f"{city_name} Metro Network Geospatial Topology Visualization (L-Space Model)", fontsize=18, pad=20)
    
    # [Critical Choice]: Setting equal axis aspect ratio ensures coordinates are not distorted, 
    # accurately reflecting the urban outline.
    plt.axis('equal') 
    plt.grid(True, linestyle=':', alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(" ----- Modeling Logic Description -----")
    print("               This project utilizes the L-Space modeling methodology from Complex Network Theory.")
    print(" [Definition]: Each metro station is abstracted as a 'node', ")
    print("           and the physical tracks directly connecting these stations are abstracted as 'edges'.")
    print(" [Rationale]: This approach provides the most direct representation of the physical topology of the metro system. ")
    print("           It serves as the foundation for analyzing network connectivity and physical path lengths, ")
    print("           as well as evaluating system performance under random failures or targeted attacks (Robustness Analysis).")

print("Environment configuration and core analytical functions have been successfully loaded.")

# ================= Cell 2: Implementation (Interactive Dropdown) =================

import ipywidgets as widgets

# Retrieve the list of cities contained in the current dataset
city_options = get_city_list(DATA_PATH)

if not city_options:
    print("No valid data detected: Please verify the data path configuration in Cell 1.")
else:
    print(f"Data loaded successfully: Retrieved metro network data for {len(city_options)} cities.")

    # Create interactive dropdown component
    dropdown = widgets.Dropdown(
        options=city_options,
        value=city_options[0],
        description='Select City:',
        style={'description_width': 'initial'}
    )

    # Bind interaction: Selecting a city will trigger real-time metric analysis and visualization
    widgets.interact(visualize_city, city_name=dropdown);

Environment configuration and core analytical functions have been successfully loaded.
Data loaded successfully: Retrieved metro network data for 48 cities.


interactive(children=(Dropdown(description='Select City:', options=('Beijing', 'Changchun', 'Changsha', 'Chang…

In [2]:
# ================= Cell 1: Core Configuration and Advanced Analytical Functions =================

# --- Theoretical Logic ---
# This code block is designed to observe metro networks through more granular dimensions.
# 1. Node Hierarchy: In the L-Space model, "Degree" represents the number of adjacent stations connected to a node.
#    - Nodes with Degree > 2 are typically junctions or multi-line transfer hubs, acting as the network's "choke points."
#    - Visual differentiation in size allows observers to immediately identify urban transportation hubs.
# 2. Robustness Fundamentals: Calculation of connected components and network diameter.
#    - A non-connected metro network implies "islands" where certain lines cannot reach others directly via rail, 
#      significantly impacting path planning efficiency.

def visualize_city_advanced(city_name):
    """
    Advanced Analysis and Visualization Function.
    Functionality: Implements node classification, line-based color coding, and provides 
    in-depth reporting on network efficiency (Diameter/Path Length).
    """
    G, edges_df = load_data(city_name)
    
    if G is None or G.number_of_nodes() == 0:
        print(f"Data Loading Failed: The dataset for {city_name} is invalid or empty.")
        return

    # --- 1. Node Hierarchy Filtering Logic ---
    # Definition: Nodes with a degree greater than 2.
    # Rationale: In standard linear segments, intermediate stations have a degree of 2.
    # A degree > 2 indicates a bifurcation point or transfer hub—the "critical few" in the network.
    transfer_stations = [n for n, d in G.degree() if d > 2]
    normal_stations = [n for n in G.nodes() if n not in transfer_stations]
    
    pos = nx.get_node_attributes(G, 'pos')
    
    print("1. Network Diameter")
    print("[Technical Definition]: Quantifies the boundary limits of the network's topological span.")
    print("[Physical Significance]: Represents the minimum Geodesic Distance between the two most distant nodes. It denotes ")
    print("           the 'worst-case scenario' for metro transit within the city.")
    print("           It defines the upper bound of accessibility under the most unfavorable paths, ")
    print("           reflecting the topological expansion cost of covering a wide geographic area.")
    print("2. Average Path Length")
    print("[Technical Definition]: Measures the global static transmission efficiency of the system.")
    print("[Physical Significance]: Reflects the average spatial cost of commuting between random station pairs. ")
    print("           It represents the overall operational efficiency of the network design in minimizing passenger travel time.")

    # --- 2. Visualization and Visual Enhancement ---
    plt.figure(figsize=(12, 10))
    ax = plt.gca()

    # (A) Render Tracks (Edges): Assign colors by line affiliation
    unique_lines = edges_df['线路'].unique()
    # Utilizing the 'tab20' colormap: Provides 20 high-contrast colors suitable for multi-line differentiation
    colors = cm.get_cmap('tab20', len(unique_lines))
    line_color_map = {line: colors(i % 20) for i, line in enumerate(unique_lines)}

    for line in unique_lines:
        line_data = edges_df[edges_df['线路'] == line]
        edge_list = list(zip(line_data['from_id'], line_data['to_id']))
        valid_edges = [(u, v) for u, v in edge_list if G.has_edge(u, v)]
        if valid_edges:
            nx.draw_networkx_edges(G, pos, edgelist=valid_edges, 
                                 edge_color=[line_color_map[line]], width=2.5, ax=ax)

    # (B) Hierarchical Node Rendering
    # Standard Stations: Smaller size with reduced alpha to serve as the background context
    nx.draw_networkx_nodes(G, pos, nodelist=normal_stations, 
                         node_size=10, node_color='black', alpha=0.4, ax=ax)
    # Hub Stations: Larger size with full opacity to highlight their core status
    nx.draw_networkx_nodes(G, pos, nodelist=transfer_stations, 
                         node_size=20, node_color='black', alpha=1.0, ax=ax)

    # (C) Legend and Aesthetics
    legend_elements = [plt.Line2D([0], [0], color=line_color_map[l], lw=3, label=l) for l in unique_lines]
    ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.01, 1), 
              title="Metro Line Legend", fontsize=10, frameon=False)
    
    plt.title(f"{city_name} Metro Topological Analysis (Hub Hierarchy Labeling)", fontsize=18, pad=20)
    plt.axis('equal') # Maintains aspect ratio to prevent geospatial distortion
    plt.grid(True, linestyle=':', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # --- 3. Computational and Professional Reporting ---
    # This section outputs quantitative evaluations of network efficiency and coverage breadth.
    
    print(f"\n--- {city_name} Metro Network In-depth Metrics Report ---")
    
    # Connectivity Analysis
    # Connectivity identifies the presence of "topological islands." 
    # Fragmentation prevents passengers from reaching all stations solely via the metro system.
    is_connected = nx.is_connected(G)
    components = nx.number_connected_components(G)
    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()
    
    print(f"[Structural Scale] Total Stations: {num_nodes} | Total Edge Segments: {num_edges}")
    print(f"[Connectivity Status] Network Connectivity: {'Optimal (Full Coverage)' if is_connected else 'Fragmented (Islands Detected)'}")
    print(f"[System Fragmentation] Independent Connected Components: {components} (Value of 1 indicates full reachability)")

    # Diameter and Average Path Length
    if is_connected:
        diameter = nx.diameter(G)
        avg_path = nx.average_shortest_path_length(G)
        print(f"[Operational Efficiency] Network Diameter: {diameter} hops (Max cost to traverse the city)")
        print(f"[Average Performance] Average Trip Length: {avg_path:.2f} hops")
    else:
        # Statistical handling: For non-connected graphs, diameter is infinite. 
        # Analysis is redirected to the Largest Connected Subgraph (LCS).
        largest_cc_nodes = max(nx.connected_components(G), key=len)
        G_sub = G.subgraph(largest_cc_nodes)
        sub_diameter = nx.diameter(G_sub)
        sub_avg_path = nx.average_shortest_path_length(G_sub)
        print(f"[Risk Alert] Note: Due to network fragmentation, the following metrics are based on the Largest Connected Subgraph (Nodes: {len(largest_cc_nodes)}):")
        print(f"    - Core Network Diameter: {sub_diameter} hops")
        print(f"    - Core Network Average Trip Length: {sub_avg_path:.2f} hops")

    print("\n--- Modeling Rationale Summary ---")
    print("This analytical block examines the metro network through refined topological dimensions:")
    print("1. [Node Hierarchy]: In the L-Space model, degree represents station adjacency.")
    print("- Nodes with Degree > 2 are typically multi-line transfer hubs or junctions (network choke points).")
    print("- Visual scaling highlights these hubs as primary urban transit nodes.")
    print("2. [Robustness Foundations]: Connectivity and diameter analysis.")
    print("- Fragmentation identifies isolated clusters, which is critical for systemic path planning and resilience.")

print("Advanced analytical functions successfully initialized.")

# ================= Cell 2: Interactive Dropdown Menu =================

import ipywidgets as widgets

# Retrieve all cities from the data directory defined in Cell 1
city_options = get_city_list(DATA_PATH)

if not city_options:
    print("No city data found. Please verify the DATA_PATH configuration in Cell 1.")
else:
    print(f"System Ready: {len(city_options)} city metro models identified.")
    
    dropdown = widgets.Dropdown(
        options=city_options,
        value=city_options[0],
        description='Select City for Analysis:',
        style={'description_width': 'initial'},
        disabled=False,
    )

    # Bind the interaction to trigger the visualize_city_advanced function
    widgets.interact(visualize_city_advanced, city_name=dropdown);

Advanced analytical functions successfully initialized.
System Ready: 48 city metro models identified.


interactive(children=(Dropdown(description='Select City for Analysis:', options=('Beijing', 'Changchun', 'Chan…

In [3]:
# ================= Cell 1: P-Space Core Configuration and Function Definitions =================

# --- Modeling Rationale ---
# The P-Space (Space P) model is fundamentally different from the previously utilized L-Space model:
# 1. Definition: In P-Space, a direct edge is established between any two stations that belong to the same metro line.
# 2. Physical Significance: Edges no longer represent physical track segments, but rather "direct reachability."
# 3. Path Interpretation: A path length of 1 signifies a direct trip, 2 denotes one required transfer, and so forth.
# 4. Rationale: This methodology directly reflects the "transfer efficiency" of the network, serving as a critical tool for evaluating passengers' perceived travel distance.

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import os
import glob
from itertools import combinations
from matplotlib import cm
import numpy as np
import warnings
import ipywidgets as widgets
from IPython.display import display

# Suppress unnecessary compatibility warnings
warnings.filterwarnings('ignore')

# -----------------------------------------------------------
# 1. Data Processing Functions: Constructing P-Space Topology
# -----------------------------------------------------------

def get_city_list(path):
    """Scans the directory to identify all available urban metro datasets"""
    if not os.path.exists(path):
        print(f"Path does not exist: {path}")
        return []
    
    files = glob.glob(os.path.join(path, "*_All_Stations.csv"))
    cities = []
    for f in files:
        filename = os.path.basename(f)
        city_name = filename.split('_All_Stations')[0]
        cities.append(city_name)
    return sorted(list(set(cities)))

def load_pspace_data(city_name):
    """
    [Core Logic]: Constructing P-Space cliques (complete subgraphs).
    All stations on the same line are fully connected pairwise to simulate direct reachability logic.
    """
    station_file = os.path.join(DATA_PATH, f"{city_name}_All_Stations.csv")
    
    if not os.path.exists(station_file):
        print(f"File missing: {city_name}")
        return None, None

    try:
        df = pd.read_csv(station_file, encoding='utf-8-sig')
        if df.empty: return None, None

        G = nx.Graph()
        
        # 1. Extract basic station information (coordinates and names)
        nodes_info = df.groupby('station ID').agg({
            '站名': 'first', '经度': 'mean', '纬度': 'mean'
        }).to_dict('index')

        for s_id, info in nodes_info.items():
            G.add_node(s_id, pos=(info['经度'], info['纬度']), label=info['站名'])

        # 2. Generate P-Space edges
        # Group by "Line" and generate pairwise combinations for all station IDs within each group
        for line, group in df.groupby('线路'):
            line_stations = group['station ID'].unique()
            # combinations(n, 2) generates the set of all possible station pairs for that line
            edges = list(combinations(line_stations, 2))
            G.add_edges_from(edges)
        
        return G, df

    except Exception as e:
        print(f"Error occurred during data retrieval or model construction for {city_name}: {e}")
        return None, None

# -----------------------------------------------------------
# 2. Deep Analysis and Visualization Functions
# -----------------------------------------------------------

def visualize_city_pspace(city_name):
    """
    Computes P-Space topological metrics and generates a high-level summarized visualization.
    Note: Computational complexity increases significantly due to the high edge count in P-Space.
    """
    G, df = load_pspace_data(city_name)
    
    if G is None or G.number_of_nodes() == 0:
        print(f"Failed to load data for {city_name}.")
        return

    # --- A. Topological Metrics and Technical Terminology ---
    print("1. Average Clustering Coefficient (ACC)")
    print("[Physical Significance]: Measures reachability redundancy and the cohesiveness of line clusters within local network regions.")
    print("           A high clustering coefficient for a station indicates that other stations sharing the same line are also highly ")
    print("           interconnected (e.g., overlapping lines).This represents lower transfer demand and stronger ")
    print("           local robustness and coverage density.")
    print("2. Closeness Centrality (CC)")
    print("[Physical Significance]: Characterizes transfer efficiency and average reachability in the global topological structure. ")
    print("           The node with the highest Closeness Centrality serves as the topological core.")
    print("           This metric quantifies a station's efficacy as a 'global transit hub'.")
    print("           Higher scores imply fewer average transfers required to reach any other random station, ")
    print("           reflecting the station's core position in system integration.")
    print(f"\n--- {city_name} Metro Network P-Space (Transfer Logic) Analysis Report ---")
    
    # 1. Connectivity Check
    is_connected = nx.is_connected(G)
    print(f"[Connectivity Level] Global reachability via metro system: {is_connected}")

    # To ensure path calculations do not fail due to network fragmentation, analyze metrics based on the Largest Connected Subgraph (LCS)
    if is_connected:
        G_main = G
    else:
        components = list(nx.connected_components(G))
        G_main = G.subgraph(max(components, key=len)).copy()
        print(f"[Data Correction] Due to isolated stations, the following path metrics are computed based on the Largest Connected Subgraph (Nodes: {G_main.number_of_nodes()}).")

    # 2. Average Clustering Coefficient
    # [Physical Significance]: Reflects the density of "direct-reach cliques." ACC in P-Space is typically very high, approaching 1.
    avg_clustering = nx.average_clustering(G)
    print(f"[Cohesiveness] Average Clustering Coefficient: {avg_clustering:.4f} (Higher values indicate broader direct-trip coverage)")

    # 3. Average Shortest Path Length (ASPL) and Diameter
    # [Conversion Logic]: Number of transfers = Path length - 1
    try:
        avg_path = nx.average_shortest_path_length(G_main)
        diameter = nx.diameter(G_main)
        print(f"[Travel Cost] Average Shortest Path Length: {avg_path:.2f} (Implies an average of {avg_path-1:.2f} transfers per trip)")
        print(f"[Maximum Cost] Network Diameter: {diameter} (Implies a maximum of {diameter-1} transfers between the two most remote stations)")
    except Exception as e:
        print(f"Calculation of path metrics timed out: {e}")

    # 4. Closeness Centrality
    # [Physical Significance]: Identifies the "Global Transfer King." High scorers require the minimum steps to reach the rest of the network.
    close_cent = nx.closeness_centrality(G)
    labels = nx.get_node_attributes(G, 'label')
    top_closeness = sorted(close_cent.items(), key=lambda x: x[1], reverse=True)[:5]
    
    print("\n[Top 5 Most Accessible Transfer Hubs] (Minimum average transfer steps to all other stations):")
    for i, (node, val) in enumerate(top_closeness):
        print(f"    Rank {i+1}: {labels[node]} (Score: {val:.4f})")

    # --- B. Visualization Logic: Dynamic Parameter Adjustment ---
    # P-Space has extremely high edge density (e.g., a 30-station line generates 435 edges).
    # Width and opacity are adjusted dynamically based on total edges (Density) to avoid visual clutter.
    plt.figure(figsize=(12, 10))
    ax = plt.gca()
    pos = nx.get_node_attributes(G, 'pos')
    
    num_edges = G.number_of_edges()
    
    # Strategy: More edges lead to thinner and more transparent lines to reveal macroscopic structure
    if num_edges < 2000:
        e_width, e_alpha = 0.4, 0.25
    elif num_edges < 5000:
        e_width, e_alpha = 0.3, 0.2
    elif num_edges < 500:
        e_width, e_alpha = 0.6, 0.45
    else:
        e_width, e_alpha = 0.25, 0.15

    # Draw background edges (reachability relations)
    nx.draw_networkx_edges(G, pos, edge_color='royalblue', width=e_width, alpha=e_alpha, ax=ax)
    
    # Draw stations using a distinct red color
    nx.draw_networkx_nodes(G, pos, node_size=10, node_color='red', alpha=0.8, ax=ax)

    # Legend configuration
    unique_lines = df['线路'].unique()
    colors = cm.get_cmap('tab20', len(unique_lines))
    legend_elements = [plt.Line2D([0], [0], marker='o', color='w', label=line, 
                                  markerfacecolor=colors(i%20), markersize=8) 
                       for i, line in enumerate(unique_lines)]
    
    # Restrict legend items to prevent overlaying the chart
    if len(legend_elements) > 25:
        legend_elements = legend_elements[:25]
        legend_elements.append(plt.Line2D([0],[0], marker='', color='w', label='...(More lines)'))

    ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.01, 1), 
              title="Line Coverage List", fontsize=9, frameon=False)

    plt.title(f"{city_name} Metro Network P-Space Topological Graph (Reachability Analysis)", fontsize=18, pad=20)
    plt.axis('equal')
    plt.grid(True, linestyle=':', alpha=0.2)
    plt.tight_layout()
    plt.show()

    print("\n--- Modeling Logic Summary ---")
    print("The P-Space model differs fundamentally from the L-Space model:")
    print("1. Definition: A direct edge exists between any two stations on the same line.")
    print("2. Physical Significance: Edges represent 'reachability' rather than physical tracks.")
    print("3. Path Interpretation: Path length represents the number of legs (Steps = Transfers + 1).")
    print("4. Rationale: Reflects transfer efficiency and topological integration from the passenger's perspective.")

print("P-Space analysis engine initialized successfully.")

# ================= Cell 2: Interactive P-Space Assessment =================

city_options = get_city_list(DATA_PATH)

if not city_options:
    print("Error: No valid city data identified.")
else:
    print(f"Ready: Currently supporting P-Space assessment for {len(city_options)} cities.")
    
    dropdown = widgets.Dropdown(
        options=city_options,
        value=city_options[0],
        description='Select City:',
        style={'description_width': 'initial'}
    )

    # Bind interaction: switching cities will trigger re-calculation and plotting
    widgets.interact(visualize_city_pspace, city_name=dropdown);

P-Space analysis engine initialized successfully.
Ready: Currently supporting P-Space assessment for 48 cities.


interactive(children=(Dropdown(description='Select City:', options=('Beijing', 'Changchun', 'Changsha', 'Chang…

In [4]:
# ================= Cell 1: P-Space Core Configuration and Robustness Analysis Functions =================

# --- Modeling Rationale and Academic Context ---
# 1. P-Space Modeling: In this framework, all stations belonging to the same line are 
#    processed as a "complete subgraph" (Clique).
#    - Physical Significance: Edges represent "direct reachability." A path length of 1 
#      indicates a trip requiring zero transfers.
#    - Rationale: It is the optimal mathematical model for evaluating the "transfer efficiency" 
#      and "logical robustness" of a metro network.
# 2. Robustness Analysis: Evaluates the risk of network collapse by simulating a "Targeted Attack."
#    - Strategy: Removing the Top 10 nodes with the highest "Degree Centrality" (i.e., the most 
#      densely connected interchange hubs).
#    - Metric: Global Efficiency. A sharp decline in efficiency post-removal indicates that the 
#      network is highly dependent on its core hub nodes.

%matplotlib inline
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import os
import glob
from itertools import combinations
from matplotlib import cm
import numpy as np
import warnings
import ipywidgets as widgets
from IPython.display import display

# Suppress unnecessary compatibility warnings to maintain a professional output environment
warnings.filterwarnings('ignore')

# -----------------------------------------------------------
# 1. Core Computational Functions
# -----------------------------------------------------------

def get_efficiency(G):
    """
    [Technical Term]: Network Global Efficiency
    [Definition]: The average of the inverse of the shortest path lengths between all node pairs.
    [Significance]: Reflects the ease of flow for passengers within the network. Values 
    approaching 1 represent superior direct reachability and lower transfer costs.
    """
    if len(G) < 2: return 0
    try:
        # Formula: E = 1 / [n(n-1)] * sum(1 / d_ij)
        return nx.global_efficiency(G)
    except:
        return 0

# -----------------------------------------------------------
# 2. In-depth Analysis and Visualization Functions
# -----------------------------------------------------------

def visualize_city_pspace_strict(city_name):
    """
    [Functionality]: Constructs the P-Space model, executes targeted attack simulations 
    on Top-10 logical hubs, and generates dynamic topological visualizations.
    """
    station_file = os.path.join(DATA_PATH, f"{city_name}_All_Stations.csv")
    
    if not os.path.exists(station_file):
        print(f"[Warning] Data file missing, unable to analyze: {city_name}")
        return

    # --- Foundational Technical Definitions ---
    print("Network Global Efficiency")
    print("[Technical Definition]: Based on P-Space topological measures, defined as the arithmetic mean of the reciprocal of ")
    print("           the shortest transfer distances between all node pairs.")
    print("[Physical Significance]: Quantifies global transfer accessibility and system integration efficiency. In the P-Space model, ")
    print("           adjacent nodes represent stations on the same line (zero transfers required).")
    print("           Thus, Global Efficiency reflects the overall capability of passengers to bypass transfer overhead when traveling ")
    print("           between random stations.Values approaching 1 signify stronger connectivity and superior performance in achieving ")
    print("           network-wide coverage with minimal transfers, indicating high topological compactness.")
    
    print("\nLogical Connection Degree")
    print("[Technical Definition]: Under the P-Space model, defined as the total number of edges associated with a single node (station), ")
    print("           equivalent to the node's degree.")
    print("[Physical Significance]: Quantifies the total volume of stations reachable from a given station without transfers.")
    print("           Stations with higher Logical Connection Degrees typically belong to lines with extensive coverage or serve as ")
    print("           major hubs connecting multiple lines. It is a core metric of 'Direct Radiation Power'; the failure of high-degree ")
    print("           nodes often leads to the fragmentation of numerous direct paths.")

    try:
        df = pd.read_csv(station_file, encoding='utf-8-sig')
        if df.empty: return

        # --- Step A: Constructing P-Space Topological Graph ---
        G_P = nx.Graph()
        nodes_info = df.groupby('station ID').agg({
            '站名': 'first', '经度': 'mean', '纬度': 'mean'
        }).to_dict('index')

        for s_id, info in nodes_info.items():
            G_P.add_node(s_id, pos=(info['经度'], info['纬度']), label=info['站名'])

        # P-Space edge logic: Shared line equals connectivity
        for line, group in df.groupby('线路'):
            line_stations = group['station ID'].unique()
            G_P.add_edges_from(combinations(line_stations, 2))

        # --- Logical Connection Degree Rankings ---
        node_degrees = dict(G_P.degree())
        # Sorting: Top 5 Highest
        top_5_high = sorted(node_degrees.items(), key=lambda x: x[1], reverse=True)[:5]
        # Sorting: Top 5 Lowest
        top_5_low = sorted(node_degrees.items(), key=lambda x: x[1], reverse=False)[:5]

        print(f"\n--- Top 5 Stations with Highest Logical Connection Degree in {city_name} ---")
        for i, (node_id, val) in enumerate(top_5_high):
            station_name = nodes_info[node_id]['站名']
            belonging_lines = df[df['station ID'] == node_id]['线路'].unique().tolist()
            print(f"Rank {i+1}: {station_name} (Degree: {val}, Line Affiliation: {belonging_lines})")

        print(f"\n--- Top 5 Stations with Lowest Logical Connection Degree in {city_name} ---")
        for i, (node_id, val) in enumerate(top_5_low):
            station_name = nodes_info[node_id]['站名']
            belonging_lines = df[df['station ID'] == node_id]['线路'].unique().tolist()
            print(f"Rank {i+1}: {station_name} (Degree: {val}, Line Affiliation: {belonging_lines})")

        # --- Step B: Auxiliary Identification of Physical Hubs (for Hierarchical Visualization) ---
        G_temp = nx.Graph()
        for line, group in df.groupby('线路'):
            line_stations = group['station ID'].unique()
            for i in range(len(line_stations)-1):
                G_temp.add_edge(line_stations[i], line_stations[i+1])
        
        transfer_stations = [n for n, d in G_temp.degree() if d > 2 and n in G_P.nodes()]
        normal_stations = [n for n in G_P.nodes() if n not in transfer_stations]

        # --- Step C: Robustness Stress Test Report ---
        print(f"\n--- In-depth Robustness Assessment of {city_name} Metro Network (P-Space) ---")
        
        degrees_p = [d for n, d in G_P.degree()]
        if degrees_p:
            print(f"[Network Characteristics] Max Logical Connectivity: {max(degrees_p)} | Min Logical Connectivity: {min(degrees_p)}")
        
        # 1. Initial State Efficiency
        print("[Analyzing] Calculating initial Network Global Efficiency...")
        initial_eff = get_efficiency(G_P)
        
        # 2. Simulated Targeted Attack: Removing the top 10 hubs (highest logical connectivity)
        top_hubs = sorted(dict(G_P.degree()).items(), key=lambda x: x[1], reverse=True)[:10]
        G_attacked = G_P.copy()
        G_attacked.remove_nodes_from([n[0] for n in top_hubs])
        
        # 3. Attacked State Efficiency Calculation
        print("[Analyzing] Calculating network efficiency after core node failures...")
        attacked_eff = get_efficiency(G_attacked)
        
        # 4. Impact Assessment
        drop_ratio = ((initial_eff - attacked_eff) / initial_eff) * 100 if initial_eff > 0 else 0

        print(f"[Core Metric] Initial Network Efficiency: {initial_eff:.4f}")
        print(f"[Core Metric] Post-Targeted Attack Efficiency: {attacked_eff:.4f}")
        print(f"[Robustness Evaluation] Efficiency Drop Ratio: {drop_ratio:.2f}% (Higher values indicate greater systemic dependency on core hubs)")
        
        labels = nx.get_node_attributes(G_P, 'label')
        print("[Critical Targets] Top-5 Removed Logical Hubs:")
        for node, val in top_hubs[:5]:
             print(f"    - {labels.get(node, node)} (Logical Connectivity: {val})")

        # --- Step D: Dynamic Visualization ---
        plt.figure(figsize=(12, 10))
        ax = plt.gca()
        pos = nx.get_node_attributes(G_P, 'pos')

        num_edges = G_P.number_of_edges()
        if num_edges < 500:
            e_width, e_alpha = 0.6, 0.45
        elif num_edges < 2000:
            e_width, e_alpha = 0.4, 0.25
        elif num_edges < 5000:
            e_width, e_alpha = 0.3, 0.2
        else:
            e_width, e_alpha = 0.15, 0.1

        nx.draw_networkx_edges(G_P, pos, edge_color='royalblue', width=e_width, alpha=e_alpha, ax=ax)

        nx.draw_networkx_nodes(G_P, pos, nodelist=normal_stations, 
                             node_size=10, node_color='red', alpha=0.4, ax=ax)
        nx.draw_networkx_nodes(G_P, pos, nodelist=transfer_stations, 
                             node_size=20, node_color='red', alpha=0.9, ax=ax)

        plt.title(f"{city_name} P-Space Transfer Logic Topology (Direct Connectivity Visualization)", fontsize=18, pad=20)
        
        unique_lines = df['线路'].unique()
        colors = cm.get_cmap('tab20', len(unique_lines))
        max_legends = 30
        legend_lines = unique_lines[:max_legends]
        legend_elements = [plt.Line2D([0], [0], marker='o', color='w', label=line, 
                                      markerfacecolor=colors(i%20), markersize=8) 
                           for i, line in enumerate(legend_lines)]
        
        if len(unique_lines) > max_legends:
            legend_elements.append(plt.Line2D([0],[0], marker='', color='w', label='... More lines'))

        ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.02, 1), 
                  title="Metro Lines Involved", fontsize=10, frameon=False)

        plt.axis('equal')
        plt.grid(True, linestyle=':', alpha=0.2)
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"[Error] Logical exception occurred while processing city {city_name}: {e}")

    # --- Summary Rationale ---
    print("\n" + "="*80)
    print("1. P-Space Modeling: In this model, all stations on the same line are treated as a 'complete subgraph' (Clique).")
    print("- Physical Significance: Edges represent 'direct reachability.' Path length 1 indicates no transfer required.")
    print("- Rationale: This is the optimal mathematical model for assessing transfer efficiency and logical robustness.")
    print("2. Robustness Analysis: Evaluates the risk of network collapse via Targeted Attack simulations.")
    print("- Strategy: Removal of the top 10 nodes with the highest Degree Centrality (densest interchange hubs).")
    print("- Evaluation Metric: Global Efficiency. A sharp decline signifies high network dependency on core hub nodes.")

print("P-Space in-depth analytical engine is now ready.")

# ================= Cell 2: Interactive Assessment System =================

city_options = get_city_list(DATA_PATH)

if not city_options:
    print("[Error] No valid city data files detected.")
else:
    print(f"[Info] The analytical environment has successfully loaded metro models for {len(city_options)} cities.")
    
    dropdown = widgets.Dropdown(
        options=city_options,
        value=city_options[0],
        description='Select City:',
        style={'description_width': 'initial'}
    )

    # Launch interaction
    widgets.interact(visualize_city_pspace_strict, city_name=dropdown);

P-Space in-depth analytical engine is now ready.
[Info] The analytical environment has successfully loaded metro models for 48 cities.


interactive(children=(Dropdown(description='Select City:', options=('Beijing', 'Changchun', 'Changsha', 'Chang…

In [5]:
# ================= Cell: Multi-city Interactive BCDE In-depth Analysis (Tables + Charts) =================

# --- Modeling Rationale ---
# This analytical block constructs a comprehensive profile of "critical nodes" within the metro network.
# We introduce the four most significant centrality metrics in complex networks (referred to as BCDE) 
# to define the "core stations" across multiple dimensions:
# 1. Degree Centrality: Measures direct connections of a station, reflecting fundamental physical connectivity.
# 2. Betweenness Centrality: Measures the frequency of a node appearing on shortest paths, reflecting its 
#    control as a "gatekeeping hub" or "choke point."
# 3. Closeness Centrality: Measures the average distance from a node to all other nodes, reflecting its 
#    efficiency as a "transit center."
# 4. Eigenvector Centrality: Measures the quality of a node's neighbors, reflecting the weighting of its 
#    structural prestige or influence.

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML

def run_full_city_analysis(city_name):
    """
    [Functionality]: Performs a comprehensive complex network analysis for a specified city.
    [Output]: Global performance report, multi-dimensional rankings, and spatial distribution visualization.
    """
    # 1. Data Loading and Preprocessing
    G, _ = load_data(city_name)
    if G is None or G.number_of_nodes() == 0:
        print(f"Error: Unable to retrieve valid data for city: {city_name}.")
        return

    # To ensure mathematical rigor for diameter and path length calculations, 
    # we must extract the Largest Connected Subgraph (LCS).
    # Rationale: Path lengths in a fragmented network are mathematically infinite.
    largest_cc = max(nx.connected_components(G), key=len)
    G_main = G.subgraph(largest_cc).copy()
    
    labels = nx.get_node_attributes(G_main, 'label')
    pos = nx.get_node_attributes(G_main, 'pos')
    
    # -------------------------------------------------------
    # 2. Computational Module: Global Metrics and Node Centrality
    # -------------------------------------------------------
    print(f"Executing in-depth topological analysis for: {city_name}...")
    
    # Global Metric Calculation
    try:
        # Diameter: Measures the "maximum geodesic cost" of traversing the network.
        diameter = nx.diameter(G_main)
        # Clustering Coefficient: Reflects the local cohesiveness of the network.
        avg_clustering = nx.average_clustering(G_main)
        # Average Shortest Path Length: The average number of hops between any station pair.
        avg_path = nx.average_shortest_path_length(G_main)
        # Density: Quantifies structural maturity (grid-like vs. radial).
        density = nx.density(G_main)
    except:
        diameter, avg_path, avg_clustering, density = "Error", "Error", "Error", "Error"

    # BCDE Centrality Computation
    deg_cent = nx.degree_centrality(G_main)      # D: Direct Connectivity
    bet_cent = nx.betweenness_centrality(G_main) # B: Gatekeeping Control
    clo_cent = nx.closeness_centrality(G_main)   # C: Global Accessibility
    try:
        # Eigenvector Centrality: Reflects node influence (nodes connected to other hubs score higher).
        eig_cent = nx.eigenvector_centrality(G_main, max_iter=2000) # E: Prestige Weighting
    except:
        eig_cent = {n: 0 for n in G_main.nodes()}

    # -------------------------------------------------------
    # 3. Output: Global Metrics Report (HTML Format)
    # -------------------------------------------------------
    display(HTML(f"<h2 style='color: #2e76b1; border-bottom: 2px solid #2e76b1;'>{city_name} Metro Network Global Performance Analysis Report</h2>"))
    
    metrics_html = f"""
    <div style="background-color: #f7f7f7; padding: 15px; border-radius: 8px; border: 1px solid #cccccc; line-height: 1.8;">
        <strong>[Network Scale]</strong> Total Stations: {len(G_main)} | Track Segments: {G_main.number_of_edges()} | Network Density: {density:.6f}<br>
        <strong>[Transmission Efficiency]</strong> Network Diameter: {diameter} hops (Max cost to traverse extreme endpoints) | Average Trip Length: {avg_path:.4f} hops<br>
        <strong>[Topological Characteristics]</strong> Average Clustering Coefficient: {avg_clustering:.4f} (Reflects local cohesiveness)
    </div>
    """
    display(HTML(metrics_html))

    # -------------------------------------------------------
    # 4. Output: Multi-dimensional Rankings (DataFrames)
    # -------------------------------------------------------
    data_rows = []
    for node in G_main.nodes():
        data_rows.append({
            'Station Name': labels.get(node, str(node)),
            'Degree Centrality': deg_cent[node],
            'Betweenness Centrality': bet_cent[node],
            'Closeness Centrality': clo_cent[node],
            'Eigenvector Centrality': eig_cent[node]
        })
    full_df = pd.DataFrame(data_rows)

    display(HTML("<h3 style='margin-top: 25px;'>Top 10 Critical Network Nodes</h3>"))
    
    # Output Top 10 for each dimension with professional descriptions
    metrics_list = [
        ('Betweenness Centrality', 'Measures gatekeeping control as a vital bridge; failure here impacts the entire system.'),
        ('Closeness Centrality', 'Measures average proximity and convenience to all network destinations.'),
        ('Degree Centrality', 'Measures direct line connectivity, reflecting base interchange capacity.'),
        ('Eigenvector Centrality', 'Measures systemic influence and structural prestige weighting.')
    ]

    for col_name, description in metrics_list:
        top_df = full_df[['Station Name', col_name]].sort_values(by=col_name, ascending=False).head(10).reset_index(drop=True)
        top_df.index = top_df.index + 1
        
        display(HTML(f"<p style='background: #e1eaf2; padding: 5px;'><strong>Dimension: {col_name}</strong> - {description}</p>"))
        display(top_df)
        print("-" * 80)

    # -------------------------------------------------------
    # 5. Visualization: 2x2 Spatial Distribution Atlas
    # -------------------------------------------------------
    metrics_map = {
        'Betweenness (Hub Control)': bet_cent,
        'Closeness (Accessibility)': clo_cent,
        'Degree (Connectivity)': deg_cent,
        'Eigenvector (Prestige)': eig_cent
    }

    fig, axes = plt.subplots(2, 2, figsize=(18, 16))
    axes = axes.flatten()
    
    # Offsets for station labels to prevent overlapping
    label_offsets = [(15, 15), (-30, 20), (25, -25), (-30, -25), (0, 35)]

    for i, (title, metric_dict) in enumerate(metrics_map.items()):
        ax = axes[i]
        nodelist = list(G_main.nodes())
        values = [metric_dict[n] for n in nodelist]
        
        # Normalization: Map node sizes from 40 to 400 to highlight hubs
        v_min, v_max = min(values), max(values)
        node_sizes = [(v - v_min)/(v_max - v_min + 1e-9) * 360 + 40 for v in values]

        # Render background edges
        nx.draw_networkx_edges(G_main, pos, alpha=0.1, edge_color='#888888', ax=ax)
        # Render nodes: Color and size represent importance scores
        sc = nx.draw_networkx_nodes(G_main, pos, nodelist=nodelist,
                                    node_size=node_sizes, node_color=values,
                                    cmap=plt.cm.viridis, alpha=0.8, ax=ax)
        
        # Colorbar configuration
        cbar = plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.04)
        cbar.ax.set_ylabel('Importance Score', fontsize=10)
        
        # Label Top 5 Stations with leader lines
        top_5_nodes = sorted(metric_dict.items(), key=lambda x: x[1], reverse=True)[:5]
        for idx, (node_id, val) in enumerate(top_5_nodes):
            x, y = pos[node_id]
            st_name = labels.get(node_id, str(node_id))
            
            ax.annotate(
                st_name,
                xy=(x, y), 
                xytext=label_offsets[idx % len(label_offsets)], 
                textcoords='offset points',
                fontsize=11, fontweight='bold',
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', boxstyle='round,pad=0.3'),
                arrowprops=dict(arrowstyle='->', connectionstyle="arc3,rad=0.1", color='#cc0000', alpha=0.6)
            )

        ax.set_title(f"{city_name}: {title}", fontsize=16, fontweight='bold', pad=15)
        ax.axis('off') # Hide coordinates to emphasize topological structure

    plt.suptitle(f"{city_name} Metro Network BCDE Centrality Distribution Atlas", fontsize=24, y=0.98)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

    print("Analytical block completed: A comprehensive profiling of 'critical nodes' in the metro network.")
    print("Introduction to the BCDE Centrality dimensions used to define core hubs:")
    print("1. Degree: Measures direct adjacency, reflecting fundamental connectivity.")
    print("2. Betweenness: Measures control over network paths, identifying gatekeeping hubs.")
    print("3. Closeness: Measures average travel distance, identifying accessibility centers.")
    print("4. Eigenvector: Measures node influence based on neighbor significance, reflecting prestige.")

# -----------------------------------------------------------
# Interactive Component: City Selection Deployment
# -----------------------------------------------------------
city_options = get_city_list(DATA_PATH)

if not city_options:
    print("No city data files found. Please verify the DATA_PATH configuration.")
else:
    # Define selection menu
    dropdown = widgets.Dropdown(
        options=city_options,
        value=city_options[0],
        description='Target City:',
        style={'description_width': 'initial'}
    )

    # Bind interaction: Full analysis report regenerates automatically upon selection
    widgets.interact(run_full_city_analysis, city_name=dropdown);

interactive(children=(Dropdown(description='Target City:', options=('Beijing', 'Changchun', 'Changsha', 'Chang…

In [6]:
# ================= Cell: Integrated Comparative Analysis of Line-level Failures (Dual-Metric Bar Chart) =================

# --- Simulation Rationale ---
# This analytical module evaluates the sensitivity of the metro network to line-level disruptions.
# 1. Simulation Process: All edges (track segments) associated with a specific line are sequentially 
#    removed from the network, while station nodes are retained.
# 2. Key Metrics:
#    - Global Efficiency (E): Reflects the loss in "travel performance." Specifically, it measures 
#      how the average topological distance between stations increases after a line is removed.
#    - Largest Connected Component (LCC) Ratio (S): Reflects the degree of "structural fragmentation." 
#      It measures the proportion of the network that remains physically connected.
# 3. Rationale: This comparative analysis distinguishes between "Efficiency Supporters" (impacting speed) 
#    and "Structural Connectors" (preventing network isolation).

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML

def simulate_line_failures_combined(city_name):
    """
    [Functionality]: Simulates metro line-level failures and integrates losses in both travel efficiency 
    and physical connectivity for comparative analysis.
    [Physical Significance]: Identifies "Critical Lines" that contribute most to the overall network performance.
    """
    # 1. Environment Setup and Baseline Establishment
    G_raw, edges_df = load_data(city_name)
    if G_raw is None or edges_df is None: 
        print(f"[Error] Unable to load data for city: {city_name}.")
        return

    # To ensure mathematical consistency of metrics, extract the initial Largest Connected Subgraph (LCS)
    lcc_nodes = max(nx.connected_components(G_raw), key=len)
    G_base = G_raw.subgraph(lcc_nodes).copy()
    N_total = G_base.number_of_nodes()
    
    # Baseline values for the initial state
    base_s = 1.0 # Initial LCC ratio defined as 100%
    base_e = nx.global_efficiency(G_base) # Baseline for travel efficiency
    
    unique_lines = edges_df['线路'].unique()
    line_results = []

    # -------------------------------------------------------
    # 2. Simulation Loop: Iterating through each line
    # -------------------------------------------------------
    for target_line in unique_lines:
        # Core Logic: Filter out edges belonging to the target line
        remaining_edges = edges_df[edges_df['线路'] != target_line]
        
        # Reconstruct the network topology post-disruption
        G_temp = nx.Graph()
        G_temp.add_nodes_from(G_base.nodes())
        for _, row in remaining_edges.iterrows():
            if row['from_id'] in G_temp and row['to_id'] in G_temp:
                G_temp.add_edge(row['from_id'], row['to_id'])
        
        # Calculate performance metrics if segments remain
        if G_temp.number_of_edges() > 0:
            # Measure LCC ratio (S Metric)
            current_lcc = len(max(nx.connected_components(G_temp), key=len)) / N_total
            # Measure Global Efficiency (E Metric)
            current_eff = nx.global_efficiency(G_temp)
        else:
            current_lcc, current_eff = 0, 0
            
        # Calculate loss percentages: higher values indicate higher line importance
        s_drop = (base_s - current_lcc) * 100
        e_drop = (base_e - current_eff) / base_e * 100
        
        line_results.append({
            "Line Name": target_line,
            "Efficiency Loss (E-Drop %)": e_drop,
            "Connectivity Loss (S-Drop %)": s_drop
        })

    # 3. Sorting Results: Ranked by Efficiency Loss to identify core lines
    df_res = pd.DataFrame(line_results).sort_values(by="Efficiency Loss (E-Drop %)", ascending=False)

    # -------------------------------------------------------
    # 4. Visualization: Grouped Bar Chart (Academic Rigor Style)
    # -------------------------------------------------------
    plt.figure(figsize=(14, 7))
    x = np.arange(len(df_res)) 
    width = 0.35 # Width of the bars

    # Render dual-metric comparison chart
    # Red indicates Efficiency Loss: reflecting increased passenger detour costs
    rects1 = plt.bar(x - width/2, df_res['Efficiency Loss (E-Drop %)'], width, 
                     label='Efficiency Loss (E)', color='#d62728', alpha=0.8) 
    # Blue indicates Connectivity Loss: reflecting network shrinkage/fragmentation
    rects2 = plt.bar(x + width/2, df_res['Connectivity Loss (S-Drop %)'], width, 
                     label='Connectivity Loss (S)', color='#1f77b4', alpha=0.6) 

    # Plot Annotations and Aesthetics
    plt.title(f"{city_name} Metro Line-level Failure Risk Analysis (Vulnerability Assessment)", fontsize=16, pad=20)
    plt.ylabel("Performance Degradation Ratio (%)", fontsize=12)
    plt.xlabel("Line Name", fontsize=12)
    plt.xticks(x, df_res['Line Name'], rotation=45, ha='right')
    plt.legend(frameon=False, fontsize=11)
    plt.grid(axis='y', linestyle=':', alpha=0.5)

    # Add data labels for small-to-medium datasets to improve readability
    if len(df_res) < 20:
        for rect in rects1:
            height = rect.get_height()
            plt.annotate(f'{height:.1f}%', xy=(rect.get_x() + rect.get_width() / 2, height),
                         xytext=(0, 3), textcoords="offset points", ha='center', fontsize=9)

    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------
    # 5. Technical Glossary and Top Rankings
    # -------------------------------------------------------
    explanation_html = """
    <div style="padding: 10px; border: 1px solid #ddd; border-left: 5px solid #2e76b1; background-color: #f9f9f9;">
        <strong>Technical Definitions:</strong><br>
        1. <strong>Efficiency Loss (E-Drop)</strong>: Measures the increase in average network transfer steps post-failure. High values indicate lines that significantly reduce global commute times.<br>
        2. <strong>Connectivity Loss (S-Drop)</strong>: Measures whether the network fragments into isolated "islands." High values indicate lines critical for maintaining topological integrity.
    </div>
    """
    display(HTML(f"<h3>Top 10 Line Failure Impacts: {city_name}</h3>"))
    display(HTML(explanation_html))
    display(df_res.head(10).reset_index(drop=True))

    print("This analytical module evaluates the sensitivity of the metro network to line-level failures.")
    print("1. Simulation Process: Sequentially removing all edges of a specific line while retaining station nodes.")
    print("2. Observed Metrics:")
    print("- Global Efficiency: Reflects travel performance loss; i.e., how much further stations are from each other on average.")
    print("- Largest Connected Component Ratio: Reflects the physical structural fragmentation of the network.")
    print("3. Rationale: This comparison identifies lines as either 'Efficiency Supporters' (impacting speed) or ")
    print("'           Structural Connectors' (preventing fragmentation).")

# --- Deployment of Interactive Component ---
city_options = get_city_list(DATA_PATH)

if not city_options:
    print("[Error] Data path not found. Please verify the DATA_PATH configuration.")
else:
    dropdown_line_comb = widgets.Dropdown(
        options=city_options, 
        value=city_options[0],
        description='Select City:',
        style={'description_width': 'initial'}
    )
    widgets.interact(simulate_line_failures_combined, city_name=dropdown_line_comb);

interactive(children=(Dropdown(description='Select City:', options=('Beijing', 'Changchun', 'Changsha', 'Chang…

In [7]:
# ================= Code Segment 1: Comprehensive Assessment and Simulation of Targeted Attacks (Based on Static Centrality) =================

import random
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import ipywidgets as widgets

# --- Global Technical Definitions (Foundational Concepts) ---
def print_robustness_basics():
    print("-" * 30 + " Foundational Concepts for Robustness Simulation " + "-" * 30)
    print("[1. Targeted Attack]")
    print("[Technical Term]: Deterministic Node Removal Strategy based on centrality ranking.")
    print("[Physical Significance]: Simulates 'intelligent' disruptive behavior equipped with global topological knowledge. ")
    print("           By prioritized stripping of critical nodes with the highest weights, it reveals the topological vulnerability of the ")
    print("           rail transit system under surgical strikes.This is a key method for identifying the impact of 'single points of failure.' ")
    print("           Nodes are purposefully removed based on their importance (e.g., transfer station weights) within the network.")
    print("\n[2. Random Failure]")
    print("[Technical Term]: Random Site Percolation process based on a uniform probability distribution.")
    print("[Physical Significance]: Simulates discrete, uncorrelated stochastic disturbances within the system (e.g., equipment aging, ")
    print("           sporadic power outages). It mimics the indiscriminate closing of stations due to daily risks. In robustness analysis, ")
    print("           Random Failure serves as the 'Baseline' to quantify the 'Acceleration Effect' of targeted attacks on structural collapse.")
    print("           It measures the marginal damage inflicted by targeted strikes compared to random disruptions.")
    print("\n[3. Percolation Theory]")
    print("[Technical Term]: The study of Phase Transitions and the evolution of the Giant Component during the removal of network components.")
    print("[Core Logic]: Investigates the critical point at which network connectivity undergoes a sudden mutation (i.e., 'collapse') ")
    print("           as nodes progressively fail.")
    print("[Physical Significance]: In the context of urban rail transit resilience, it defines the upper bound of the damage threshold ")
    print("           the system can tolerate before cascading failure.This theory is utilized to detect the critical dynamics of the transition ")
    print("           from a 'globally connected state' to a 'discrete fragmented state.' It helps determine the maximum damage ratio ")
    print("           a transit grid can sustain before total paralysis")
    print("\n[4. Proportion of the Largest Connected Component (S)]")
    print("[Technical Term]: The Order Parameter used to quantify the macroscopic topological integrity of a system.")
    print("[Physical Significance]: S is defined as the ratio of nodes in the Largest Connected Subgraph (Giant Component) to the total number of nodes ")
    print("           in the original network. It characterizes the scale of the backbone structure capable of maintaining cross-regional ")
    print("           accessibility after disturbances. A step-like decline in S signifies that the network has reached a structural phase ")
    print("           transition point leading to total fragmentation.")
    print("-" * 84 + "\n")

def simulate_robustness(city_name):
    """
    [Functionality]: Simulates the collapse process of the metro network under targeted attacks and random failures.
    [Features]: Uses a single-run random failure as the baseline and outputs detailed comparative data at key percentages.
    """
    # 1. Primary Data Preparation
    G_orig, _ = load_data(city_name)
    if G_orig is None: return
    
    # Extract the initial Largest Connected Component (LCC) to ensure a 100% physically connected starting point.
    lcc_nodes = max(nx.connected_components(G_orig), key=len)
    G = G_orig.subgraph(lcc_nodes).copy()
    N = G.number_of_nodes()
    node_labels = nx.get_node_attributes(G, 'label')
    
    # -----------------------------------------------------------
    # 2. Internal Function: Network Collapse Simulator
    # -----------------------------------------------------------
    def get_collapse_data(node_list):
        temp_G = G.copy()
        s_curve = [1.0] 
        removed_nodes_info = []
        threshold_stats = {}
        
        for i in range(1, N + 1):
            node_to_remove = node_list[i-1]
            if temp_G.has_node(node_to_remove):
                temp_G.remove_node(node_to_remove)
            
            if i <= 10: 
                removed_nodes_info.append(node_labels.get(node_to_remove, str(node_to_remove)))
            
            if temp_G.number_of_nodes() > 0:
                largest_cc_size = len(max(nx.connected_components(temp_G), key=len))
                s = largest_cc_size / N
            else:
                s = 0
            s_curve.append(s)
            
            # Log survival metrics at critical damage percentages
            for pct in [5, 10, 25, 50, 75]:
                if i == int(N * pct / 100): 
                    threshold_stats[f"{pct}%"] = s
        return s_curve, removed_nodes_info, threshold_stats

    # --- Step 1: Establish Single-run Random Failure Baseline ---
    print(f"--- Executing Robustness Stress Test for {city_name} (Single-run Random Failure Baseline) ---")
    nodes_rand = list(G.nodes())
    random.shuffle(nodes_rand)
    # Obtain full curve and key-point statistics for a single stochastic failure event
    random_curve, _, random_stats_data = get_collapse_data(nodes_rand)

    # --- Step 2: Comparative Analysis of Targeted Attacks across BCDE Centralities ---
    for attack_metric in ['Betweenness', 'Degree', 'Closeness', 'Eigenvector']:
        
        if attack_metric == 'Betweenness': 
            cent = nx.betweenness_centrality(G)
        elif attack_metric == 'Degree': 
            cent = nx.degree_centrality(G)
        elif attack_metric == 'Closeness': 
            cent = nx.closeness_centrality(G)
        elif attack_metric == 'Eigenvector':
            try: cent = nx.eigenvector_centrality(G, max_iter=2000)
            except: cent = nx.degree_centrality(G)
        
        # Generate the targeted attack sequence ordered by decreasing importance
        nodes_targeted = sorted(cent, key=cent.get, reverse=True)
        targeted_curve, top_nodes, targeted_stats = get_collapse_data(nodes_targeted)

        # --- Step 3: Output Analysis Report (Targeted S vs. Random S Comparison) ---
        print(f"\n[{attack_metric} Targeted Attack Analysis Report]")
        print(f"1. Core Failure Sequence (Top 10): {' -> '.join(top_nodes)}")
        print(f"2. Percolation Threshold Analysis (LCC Proportion $S$ Comparison):")
        for pct in ['5%', '10%', '25%', '50%', '75%']:
            t_s = targeted_stats.get(pct, 0.0)
            r_s = random_stats_data.get(pct, 0.0)
            print(f"   - Post {pct} node removal: Targeted Strike S = {t_s:.4f} | Random Failure S = {r_s:.4f}")

        # --- Step 4: Visualization Rendering ---
        plt.figure(figsize=(10, 5))
        x_axis = np.array(range(len(targeted_curve))) / N * 100
        
        plt.plot(x_axis, random_curve, label='Random Failure (Single Run Baseline)', 
                 color='green', lw=2, linestyle='--', alpha=0.6)
        plt.plot(x_axis, targeted_curve, label=f'Targeted Attack ({attack_metric})', 
                 color='red', lw=2.5)
        
        plt.title(f"Percolation Analysis: {city_name} ({attack_metric})", fontsize=14, fontweight='bold')
        plt.xlabel("Percentage of Nodes Removed (%)", fontsize=12)
        plt.ylabel("LCC Proportion ($S$)", fontsize=12)
        plt.xlim(0, 100) 
        plt.ylim(0, 1.05)
        plt.grid(True, linestyle=':', alpha=0.5)
        plt.legend(frameon=False)
        plt.tight_layout()
        plt.show()
        print("-" * 92)

    # --- Final Protocol Evaluation and Academic Interpretation ---
    print("\n[Protocol Evaluation]")
    print("Advantages:")
    print("1. High Metric Specificity: By comparing four topological attributes, ")
    print("           it precisely locates node types that contribute most to network survival.")
    print("2. Objective Reference Value: The random failure curve provides an objective benchmark for assessing ")
    print("           the severity of intentional disruptions.")
    print("3. Real-time Data Comparison: The attack report synchronizes S-values of targeted vs. ")
    print("           random scenarios to quantify the attack acceleration effect.")
    print("\nLimitations:")
    print("1. Static Evaluation Constraints: The attack sequence is fixed before simulation begins; ")
    print("           it does not account for the complexity of dynamic reassessment after network damage.")
    print("2. Stochastic Variance: Since this uses a single random simulation run, ")
    print("           the random baseline may be influenced by accidental removal sequences, ")
    print("           resulting in slightly lower rigor than a multi-run average.")
    
    print("\n[Academic Interpretation of Trend Analysis]")
    print("Interpretation: If the red solid line (Targeted Attack) descends significantly faster than the green dashed line (Random Failure), ")
    print("           the network exhibits the 'Robust yet Fragile' property.")
    print("This property indicates that while the system is resilient to common random failures, ")
    print("           it is extremely sensitive to surgical strikes on its topological hubs.")

# --- Interactive Interface Deployment ---
city_list = get_city_list(DATA_PATH)
if city_list:
    print_robustness_basics()
    widgets.interact(
        simulate_robustness, 
        city_name=widgets.Dropdown(options=city_list, description='Select City:')
    )

------------------------------ Foundational Concepts for Robustness Simulation ------------------------------
[1. Targeted Attack]
[Technical Term]: Deterministic Node Removal Strategy based on centrality ranking.
[Physical Significance]: Simulates 'intelligent' disruptive behavior equipped with global topological knowledge. 
           By prioritized stripping of critical nodes with the highest weights, it reveals the topological vulnerability of the 
           rail transit system under surgical strikes.This is a key method for identifying the impact of 'single points of failure.' 
           Nodes are purposefully removed based on their importance (e.g., transfer station weights) within the network.

[2. Random Failure]
[Technical Term]: Random Site Percolation process based on a uniform probability distribution.
[Physical Significance]: Simulates discrete, uncorrelated stochastic disturbances within the system (e.g., equipment aging, 
           sporadic power outages). It mimics t

interactive(children=(Dropdown(description='Select City:', options=('Beijing', 'Changchun', 'Changsha', 'Chang…

In [8]:
# ================= Code Segment 2: Comprehensive Robustness Assessment System (Full-Threshold Snapshot Comparison) =================

import random
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import ipywidgets as widgets
from matplotlib import cm

# --- 1. Global Technical Definitions (Academic Audit Version) ---
def print_terminology_report():
    print("-" * 30 + " Technical Terminology and Parameter Definitions " + "-" * 30)
    print("[1. Macroscopic Topological Integrity (Size of the Largest Connected Component, S)]")
    print(" [Technical Definition]: The ratio of the number of nodes in the Largest Connected Component (LCC) ")
    print("          to the total nodes in the original network.")
    print(" [Physical Significance]: As the Order Parameter in Percolation Theory, the S-value quantifies the ")
    print("          macroscopic connectivity level of the metro system's physical backbone.")
    print("          It characterizes the scale of the skeleton structure capable of maintaining cross-regional accessibility after disturbances.")
    print("          A sudden drop in S (breaking the critical threshold) signifies a structural phase transition, ")
    print("          indicating the system has reached total fragmentation.")
    print("\n[2. Global Operational Efficiency (Global Efficiency, E)]")
    print("[Technical Definition]: The arithmetic mean of the reciprocal of the shortest path lengths between all node pairs.")
    print("[Physical Significance]: A measure of system functional efficacy, reflecting performance degradation during network disruption.")
    print("          Since this metric incorporates the inverse of path lengths, it effectively evaluates the average reachability of ")
    print("          the residual structure for passengers even after the network splits into multiple islands.  ")
    print("          While the physical skeleton (S) might persist, the failure of core hubs can lead to exponential increases in detour costs, ")
    print("          which is reflected by a significant drop in E.")
    print("\n[3. Cumulative Robustness Index (Robustness Index, R)]")
    print("[Technical Definition]: The cumulative integrated area under the performance decay curve (S or E metrics) ")
    print("          throughout the node removal process, also known as the Area Under Curve (AUC).")
    print(" [Physical Significance]: Quantifies the average survival efficacy of the system during a 'continuous strike.' ")
    print("          The R-value integrates instantaneous collapse points and long-term decay into a scalar value within the range [0, 0.5].")
    print("          Higher values represent superior topological robustness and resilience against continuous removal pressure.")
    print("\n[4. Targeted Attack vs. Random Failure]")
    print("[Random Failure]: A node percolation process based on a uniform probability distribution.")
    print("          Aims to simulate discrete stochastic accidents (e.g., equipment aging, localized power failures), ")
    print("          serving as the performance 'Baseline' for intrinsic redundancy.")
    print("[Targeted Attack]: A deterministic node removal strategy based on centrality rankings (e.g., Betweenness or Degree).")
    print("          Designed to simulate precise destruction of core hubs to identify topological vulnerabilities.")
    print("-" * 84 + "\n")

def simulate_robustness_pro(city_name, num_random_runs=20):
    """
    Executes in-depth robustness simulations and outputs a comprehensive report 
    including technical explanations, protocol assessments, and anomaly analysis.
    """
    # 1. Initial Data Preprocessing
    G_orig, _ = load_data(city_name)
    if G_orig is None: return
    
    # Extract the Largest Connected Component as the analysis baseline
    lcc_nodes = max(nx.connected_components(G_orig), key=len)
    G = G_orig.subgraph(lcc_nodes).copy()
    N = G.number_of_nodes()
    initial_efficiency = nx.global_efficiency(G)
    
    # Set removal step size (1% intervals)
    step = max(1, int(N * 0.01))
    removal_steps = list(range(0, N + 1, step)) 
    if removal_steps[-1] != N: removal_steps.append(N)

    def run_attack_sim(node_list):
        """Internal Simulation Engine: Removes nodes sequentially and logs topological metrics."""
        s_list, e_list = [1.0], [initial_efficiency]
        temp_G = G.copy()
        current_removed = 0
        snapshot_data = {} # Stores data at critical percentage thresholds

        for target_remove in removal_steps[1:]:
            while current_removed < target_remove and current_removed < len(node_list):
                node_to_del = node_list[current_removed]
                if temp_G.has_node(node_to_del):
                    temp_G.remove_node(node_to_del)
                current_removed += 1
            
            # Compute post-disruption metrics
            if temp_G.number_of_nodes() > 1:
                s_val = len(max(nx.connected_components(temp_G), key=len)) / N
                e_val = nx.global_efficiency(temp_G)
            else:
                s_val, e_val = 0, 0
            
            s_list.append(s_val); e_list.append(e_val)

            # Capture snapshots at specific thresholds
            current_pct = round(current_removed / N * 100)
            if current_pct in [5, 10, 25, 50, 75] and current_pct not in snapshot_data:
                snapshot_data[current_pct] = {'S': s_val, 'E': e_val}

        return np.array(s_list), np.array(e_list), snapshot_data

    # 2. Establish Baseline: Random Failure Simulation (Monte Carlo Sampling)
    print(f"--- Comprehensive Robustness Assessment Report: {city_name} Metro Network ---")
    print(f"[Execution] Performing {num_random_runs} Monte Carlo iterations for statistical baseline...")
    
    s_rand_all, e_rand_all = [], []
    random_snapshots_collection = [] 

    for _ in range(num_random_runs):
        nodes_rand = list(G.nodes()); random.shuffle(nodes_rand)
        s, e, snaps = run_attack_sim(nodes_rand)
        s_rand_all.append(s); e_rand_all.append(e)
        random_snapshots_collection.append(snaps)
    
    s_random_mean, s_random_std = np.mean(s_rand_all, axis=0), np.std(s_rand_all, axis=0)
    e_random_mean, e_random_std = np.mean(e_rand_all, axis=0), np.std(e_rand_all, axis=0)

    # Calculate mean performance of random failures at thresholds
    random_threshold_means = {}
    for pct in [5, 10, 25, 50, 75]:
        s_vals = [d[pct]['S'] for d in random_snapshots_collection if pct in d]
        e_vals = [d[pct]['E'] for d in random_snapshots_collection if pct in d]
        random_threshold_means[pct] = {'S': np.mean(s_vals) if s_vals else 0, 
                                        'E': np.mean(e_vals) if e_vals else 0}

    # 3. Strategic Attack Simulations: BCDE Centrality Comparison
    for attack_metric in ['Betweenness', 'Degree', 'Closeness', 'Eigenvector']:
        # Compute node importance weights
        if attack_metric == 'Betweenness': cent = nx.betweenness_centrality(G)
        elif attack_metric == 'Degree': cent = nx.degree_centrality(G)
        elif attack_metric == 'Closeness': cent = nx.closeness_centrality(G)
        else:
            try: cent = nx.eigenvector_centrality(G, max_iter=1000)
            except: cent = nx.degree_centrality(G)
        
        # Generate attack sequence in descending order of weight
        nodes_targeted = sorted(cent, key=cent.get, reverse=True)
        s_targeted, e_targeted, targeted_snaps = run_attack_sim(nodes_targeted)

        # Calculate Robustness R-Indices (Normalized AUC)
        r_s = np.trapz(s_targeted) / len(s_targeted)
        r_e = np.trapz(e_targeted) / len(e_targeted)

        # Output real-time analysis for the city
        print(f"--- Robustness Analysis Summary: {city_name} ---")

        # Attack Efficacy Snapshot Report
        print(f"\n[Attack Efficacy Snapshot: Comparison at 5%/10%/25%/50%/75% Node Removal via {attack_metric}]")
        print(f"{'Removal Ratio':<15} | {'Targeted S':<12} | {'Random Mean S':<15} | {'Targeted E':<12} | {'Random Mean E':<15}")
        print("-" * 85)
        for pct in [5, 10, 25, 50, 75]:
            t_s = targeted_snaps.get(pct, {'S': 0})['S']
            t_e = targeted_snaps.get(pct, {'E': 0})['E']
            r_s_base = random_threshold_means.get(pct, {'S': 0})['S']
            r_e_base = random_threshold_means.get(pct, {'E': 0})['E']
            print(f"{pct}% Removed    | {t_s:.4f}       | {r_s_base:.4f}       | {t_e:.4f}       | {r_e_base:.4f}")

        print("\n[Systemic Robustness Metrics Interpretation]")
        print(f"1. Structural Robustness Index (R_s = {r_s:.3f}): Reflects the rate of physical fragmentation when critical nodes (via {attack_metric}) are removed.")
        print(f"2. Functional Robustness Index (R_e = {r_e:.3f}): Reflects the decay dynamics of global commuting efficiency post-core-node failure.")

        # 4. Statistical Visualization
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
        x_axis = np.array(removal_steps) / N * 100 
        
        # S-Metric Collapse Curve
        ax1.plot(x_axis, s_targeted, color='red', marker='o', label=f'Targeted Attack ({attack_metric})', markersize=3)
        ax1.plot(x_axis, s_random_mean, color='green', label='Random Failure (Mean)')
        ax1.fill_between(x_axis, np.maximum(0, s_random_mean - s_random_std), s_random_mean + s_random_std, color='green', alpha=0.1)
        ax1.set_title(f"Structural Integrity (S)\nRobustness Index R: {r_s:.3f}"); ax1.set_ylim(0, 1.05)
        ax1.set_xlabel("Percentage of Nodes Removed (%)"); ax1.set_ylabel("Giant Component Ratio"); ax1.legend()

        # E-Metric Decay Curve
        ax2.plot(x_axis, e_targeted, color='red', marker='o', label=f'Targeted Attack ({attack_metric})', markersize=3)
        ax2.plot(x_axis, e_random_mean, color='green', label='Random Failure (Mean)')
        ax2.fill_between(x_axis, np.maximum(0, e_random_mean - e_random_std), e_random_mean + e_random_std, color='green', alpha=0.1)
        ax2.set_title(f"Operational Efficiency (E)\nRobustness Index R: {r_e:.3f}"); ax2.set_ylim(0, initial_efficiency * 1.1)
        ax2.set_xlabel("Percentage of Nodes Removed (%)"); ax2.set_ylabel("Global Efficiency"); ax2.legend()
        
        plt.tight_layout()
        plt.show()
        print("-" * 85)


    # 5. Evaluation of Model Strengths and Academic Limitations
    print("\n[Protocol Assessment and Model Evaluation]")
    print("1. Core Strengths: Integration of multi-run Monte Carlo simulations effectively isolates stochastic noise. ")
    print("          Use of AUC integration enables standardized comparison of heterogeneous network robustness.")
    print("2. Academic Limitations: The simulation currently employs a non-adaptive attack strategy based on initial static topological rankings;") 
    print("          it does not yet model the dynamic redistribution of passenger flows over the residual topology during network collapse.")
    print("\n[Technical Note on Critical Anomalies]")
    print("Observation: When the node removal ratio is extremely high (>90%), the E-curve may exhibit a non-physical data uptick.")
    print("Mechanism: This is a 'Small-Sample Bias' inherent in complex network metrics. When a network is degraded to a few remaining nodes ")
    print("          (e.g., 2-3) that happen to be directly connected, the global efficiency formula yields a very short path length, ")
    print("          creating a mathematical artifact of 'increased efficiency.'")
    print("Conclusion: This localized uptick lacks significance in transit planning. Robustness assessments should prioritize the ")
    print("          slope changes and collapse thresholds during the early-to-mid stages of the curve.")

# --- Interactive System Launch ---
city_options = get_city_list(DATA_PATH)
if city_options:
    print_terminology_report()
    widgets.interact(
        simulate_robustness_pro, 
        city_name=widgets.Dropdown(options=city_options, description='Select City:'),
        num_random_runs=widgets.Dropdown(options=[10, 20, 50], value=20, description='MC Iterations:')
    )

------------------------------ Technical Terminology and Parameter Definitions ------------------------------
[1. Macroscopic Topological Integrity (Size of the Largest Connected Component, S)]
 [Technical Definition]: The ratio of the number of nodes in the Largest Connected Component (LCC) 
          to the total nodes in the original network.
 [Physical Significance]: As the Order Parameter in Percolation Theory, the S-value quantifies the 
          macroscopic connectivity level of the metro system's physical backbone.
          It characterizes the scale of the skeleton structure capable of maintaining cross-regional accessibility after disturbances.
          A sudden drop in S (breaking the critical threshold) signifies a structural phase transition, 
          indicating the system has reached total fragmentation.

[2. Global Operational Efficiency (Global Efficiency, E)]
[Technical Definition]: The arithmetic mean of the reciprocal of the shortest path lengths between all nod

interactive(children=(Dropdown(description='Select City:', options=('Beijing', 'Changchun', 'Changsha', 'Chang…

In [9]:
# ================= Code Segment 3: Adaptive Robustness Multi-dimensional Analysis (Enhanced Academic Version) =================

import random
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display, HTML
import ipywidgets as widgets

# --- 1. Global Academic Glossary and Advanced Parameter Definitions ---
def print_comprehensive_theory_report():
    print("-" * 30 + " Dictionary of Complex Network Robustness Theory " + "-" * 30)

    print("[1. Macroscopic Structural Connectivity (Size of the Giant Component, S)]")
    print("[Technical Definition]: The ratio of the number of nodes in the Largest Connected Component (LCC) to the total original network size.")
    print("[Physical Significance]: Serving as the Order Parameter in Percolation Theory, it directly reflects whether the 'physical skeleton' ")
    print("          of the metro system maintains global-scale spatial reachability.")
    print("          It measures the proportion of the largest mutually connected cluster relative to the original scale.")

    print("\n[2. Global Transmission Efficiency (Global Efficiency, E)]")
    print("[Technical Definition]: The arithmetic mean of the reciprocal of the shortest path lengths between all node pairs in the network.")
    print("[Physical Significance]: Quantifies the 'Service Efficacy' of the system. Even if the physical structure has not fully fragmented, ")
    print("          the failure of core hubs can cause an exponential jump in detour costs, ")
    print("          leading to a rapid collapse of E and signifying a loss of transit service quality.")

    print("\n[3. Critical Collapse Threshold (Critical Threshold, fc)]")
    print("[Technical Definition]: In this experiment, the 'Collapse Threshold' is defined as the point where S < 0.1 ")
    print("          (i.e., the LCC drops below 10% of the original size). It represents the node removal fraction at this stage.")
    print("[Rationale]: Based on Percolation Theory, once a system loses 90% of its reachability, ")
    print("          its function as a critical social infrastructure is considered to have completely failed.")
    print("          The 0.1 threshold serves as a standardized benchmark, enabling a fair comparison of resilience limits ")
    print("          across cities of varying scales.")

    print("\n[4. Adaptive Attack Logic (Adaptive Attack Strategy)]")
    print("[Algorithmic Definition]: Centrality rankings of the residual network are recalculated in real-time ")
    print("          after each node removal step to determine the next target.")
    print("[Academic Value]: Simulates extreme disruptive behavior equipped with real-time dynamic intelligence. ")
    print("          Compared to traditional static attacks, this mode more accurately reveals the risks of cascading failure ")
    print("          during network topological reconstruction.")
    print("-" * 85 + "\n")

# --- 2. Core Simulation Function ---
def simulate_robustness_adaptive_pro(city_name, num_random_runs=20):
    """
    Executes in-depth adaptive robustness simulations and generates a comprehensive academic report 
    including technical terminology, BCDE strategy analysis, and multi-threshold snapshot comparisons.
    """
    # Load Data
    G_orig, _ = load_data(city_name)
    if G_orig is None: return
    lcc_nodes = max(nx.connected_components(G_orig), key=len)
    G = G_orig.subgraph(lcc_nodes).copy()
    N = G.number_of_nodes()
    initial_eff = nx.global_efficiency(G)
    
    metrics = ['Degree', 'Betweenness', 'Closeness', 'Eigenvector']

    def get_centrality(graph, method):
        if method == 'Betweenness': return nx.betweenness_centrality(graph)
        if method == 'Degree': return nx.degree_centrality(graph)
        if method == 'Closeness': return nx.closeness_centrality(graph)
        if method == 'Eigenvector':
            try: return nx.eigenvector_centrality(graph, max_iter=500)
            except: return nx.degree_centrality(graph)
        return nx.degree_centrality(graph)

    # Define 1% step-size sampling points
    step_size = max(1, int(N * 0.01))
    removal_points = list(range(0, N + 1, step_size))
    if removal_points[-1] != N: removal_points.append(N)
    x_axis_pct = np.array(removal_points) / N * 100

    def run_sim(mode, metric_name=None, static_nodes=None):
        temp_G = G.copy()
        s_list, e_list = [1.0], [initial_eff]
        current_removed = 0
        snapshot_data = {} 

        for target_remove in removal_points[1:]:
            num_to_rem = target_remove - current_removed
            if mode == 'Adaptive' and temp_G.number_of_nodes() > 0:
                curr_cent = get_centrality(temp_G, metric_name)
                target_nodes = sorted(curr_cent, key=curr_cent.get, reverse=True)
            elif mode == 'Static':
                target_nodes = [n for n in static_nodes if n in temp_G]
            else: # Random Failure
                target_nodes = list(temp_G.nodes()); random.shuffle(target_nodes)

            nodes_to_del = target_nodes[:num_to_rem]
            temp_G.remove_nodes_from(nodes_to_del)
            current_removed += len(nodes_to_del)

            if temp_G.number_of_nodes() > 1:
                l_cc = len(max(nx.connected_components(temp_G), key=len)) / N
                eff = nx.global_efficiency(temp_G)
            else:
                l_cc, eff = 0, 0
            
            s_list.append(l_cc); e_list.append(eff)

            # Capture key thresholds for reporting
            pct_current = round(current_removed / N * 100)
            if pct_current in [5, 10, 25, 50, 75] and pct_current not in snapshot_data:
                snapshot_data[pct_current] = (l_cc, eff)

        return np.array(s_list), np.array(e_list), snapshot_data

    # --- Phase A: Protocol Assessment ---
    print(f"--- Adaptive Robustness Stress Test Report: {city_name} Network ---")
    print("\n[Experimental Design Evaluation]")
    print("Key Strengths:")
    print("1. Dynamic Reconstruction: Utilizing adaptive recalculation accurately captures the migration of importance during network degradation.")
    print("2. Statistical Rigor: Multi-run Monte Carlo sampling establishes a random failure baseline (green shaded area), ")
    print("          ensuring statistical significance.")
    print("3. Boundary Correction: The use of clipping logic prevents negative artifacts caused by statistical fluctuations at the plot boundaries.")
    print("\n[Academic Limitations]")
    print("1. Computational Complexity: Adaptive recalculation (especially Betweenness) is computationally intensive for mega-scale metro networks.")
    print("2. Traffic Load Constraints: The current model focuses on topological resilience and does not yet ")
    print("          integrate dynamic Traffic Redistribution effects.")

    # --- Phase B: Statistical Baseline Construction ---
    print(f"\n[Status] Executing {num_random_runs} Monte Carlo iterations for statistical baseline...")
    s_rand_all, e_rand_all = [], []
    random_snapshots_collection = []
    for _ in range(num_random_runs):
        sr, er, snap = run_sim('Random')
        s_rand_all.append(sr); e_rand_all.append(er)
        random_snapshots_collection.append(snap)
    
    s_rand_mean, s_rand_std = np.mean(s_rand_all, axis=0), np.std(s_rand_all, axis=0)
    e_rand_mean, e_rand_std = np.mean(e_rand_all, axis=0), np.std(e_rand_all, axis=0)

    # Pre-processing mean random failure values at thresholds
    random_base_table = {}
    for pct in [5, 10, 25, 50, 75]:
        s_vals = [d[pct][0] for d in random_snapshots_collection if pct in d]
        e_vals = [d[pct][1] for d in random_snapshots_collection if pct in d]
        random_base_table[pct] = (np.mean(s_vals), np.mean(e_vals))

    summary_list = []

    # --- Phase C: BCDE Targeted Attack Analysis ---
    for metric in metrics:
        print(f"\n{'-' * 10} Deep Strategy Analysis: Targeted Attack based on {metric} {'-' * 10}")
        
        # Academic Strategy Interpretation
        if metric == 'Degree':
            print("[Strategy Bias]: Connectivity Breadth Attack. Target interchange hubs with high physical adjacency to paralyze local aggregation.")
        elif metric == 'Betweenness':
            print("[Strategy Bias]: Hub Choke-point Attack. Target bridge nodes carrying the most paths to sever cross-regional long-distance reachability.")
        elif metric == 'Closeness':
            print("[Strategy Bias]: Topological Centroid Attack. Target nodes with minimum average distance to maximize network detour costs.")
        else:
            print("[Strategy Bias]: Influence Prestige Attack. Target prestigious nodes connected to other critical cores.")

        static_cent = get_centrality(G, metric)
        nodes_static = sorted(static_cent, key=static_cent.get, reverse=True)
        
        # Comparative Experiments
        s_adaptive, e_adaptive, adaptive_snap = run_sim('Adaptive', metric)
        s_static, e_static, _ = run_sim('Static', static_nodes=nodes_static)
        
        # Identifying Adaptive fc (S < 0.1)
        fc_val = 100.0
        for idx, val in enumerate(s_adaptive):
            if val < 0.1: fc_val = x_axis_pct[idx]; break

        # --- Output Multi-stage Performance Snapshot Comparison ---
        print(f"\n[{metric} Adaptive Attack: Multi-stage Performance Snapshot]")
        print(f"{'Removal %':<10} | {'Adap S':<10} | {'Rand S':<10} | {'Adap E':<10} | {'Rand E':<10}")
        print("-" * 75)
        for pct in [5, 10, 25, 50, 75]:
            as_val, ae_val = adaptive_snap.get(pct, (0, 0))
            rs_base, re_base = random_base_table.get(pct, (0, 0))
            print(f"{pct}% Removed | {as_val:.4f}     | {rs_base:.4f}     | {ae_val:.4f}     | {re_base:.4f}")

        # 4. Visualization (with boundary clipping)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
        
        # S-Curve: Connectivity Collapse
        ax1.plot(x_axis_pct, s_adaptive, color='red', marker='o', label=f'Adaptive {metric}', markersize=3)
        ax1.plot(x_axis_pct, s_static, color='orange', linestyle='--', label='Static Attack', alpha=0.7)
        ax1.plot(x_axis_pct, s_rand_mean, color='green', label='Random Baseline')
        ax1.fill_between(x_axis_pct, np.maximum(0, s_rand_mean - s_rand_std), 
                         s_rand_mean + s_rand_std, color='green', alpha=0.1)
        ax1.axvline(x=fc_val, color='red', linestyle=':', label=f'Adaptive fc: {fc_val:.1f}%')
        ax1.set_title(f"Connectivity Decay (S)\nAdaptive fc: {fc_val:.1f}%", fontweight='bold')
        ax1.set_ylim(0, 1.05); ax1.set_xlabel("Nodes Removed (%)"); ax1.legend(frameon=False)

        # E-Curve: Efficiency Decay
        ax2.plot(x_axis_pct, e_adaptive, color='blue', marker='o', label='Adaptive Efficiency', markersize=3)
        ax2.plot(x_axis_pct, e_rand_mean, color='green', label='Random Efficiency')
        ax2.fill_between(x_axis_pct, np.maximum(0, e_rand_mean - e_rand_std), 
                         e_rand_mean + e_rand_std, color='green', alpha=0.1)
        ax2.set_title(f"Efficiency Decay (E)\nInitial E: {initial_eff:.4f}", fontweight='bold')
        ax2.set_ylim(0, initial_eff * 1.1); ax2.set_xlabel("Nodes Removed (%)"); ax2.legend(frameon=False)
        
        plt.suptitle(f"{city_name} Adaptive Robustness Stress Test: {metric}", fontsize=18, y=1.05)
        plt.tight_layout(); plt.show()
        print("-" * 92)

        summary_list.append({
            "Metric": metric, "Adaptive Collapse Point (fc)": f"{fc_val:.2f}%",
            "Adaptive Robustness (R)": f"{np.trapz(s_adaptive)/len(s_adaptive):.4f}",
            "Static Robustness (R)": f"{np.trapz(s_static)/len(s_static):.4f}"
        })

    # --- Phase D: Conclusion and Synthesis ---
    print("\n" + "="*30 + " Synthesis of Academic Observations " + "="*30)
    print("[1. Divergence in Decay Rates of S and E]")
    print("   Observation: E typically decays significantly faster than S.")
    print("   Mechanism: This indicates that 'Service Efficacy' suffers pre-emptive degradation. Before large-scale physical fragmentation occurs,")
    print("          average commute costs surge due to the severance of critical shortest paths.")
    print("\n[2. Technical Note on Terminal Anomaly (Efficiency Uptick)]")
    print("   Note: Upticks at >90% removal are computational artifacts of 'Small-Sample Bias.'")
    print("          When the network consists of only 2-3 remaining connected nodes, the denominator effect creates a deceptive 'efficiency peak' ")
    print("          with no practical transit planning value.")
    print("\n[3. Static vs. Adaptive Attack Divergence]")
    print("   Observation: The Adaptive curve (Red) consistently sits below the Static curve (Orange).")
    print("   Inference: This reveals the presence of 'Hidden Hubs.' As primary hubs are removed, secondary nodes rapidly assume critical roles. ")
    print("          Adaptive attacks more effectively compromise systemic resilience by targeting these migrating hubs.")
    print(f"\n--- Quantitative Summary of Adaptive Robustness: {city_name} ---")
    display(pd.DataFrame(summary_list))

# --- Deployment ---
city_list = get_city_list(DATA_PATH)
if city_list:
    print_comprehensive_theory_report()
    widgets.interact(
        simulate_robustness_adaptive_pro, 
        city_name=widgets.Dropdown(options=city_list, description='Target City:'),
        num_random_runs=widgets.Dropdown(options=[10, 20, 50], value=20, description='Iterations:')
    )

------------------------------ Dictionary of Complex Network Robustness Theory ------------------------------
[1. Macroscopic Structural Connectivity (Size of the Giant Component, S)]
[Technical Definition]: The ratio of the number of nodes in the Largest Connected Component (LCC) to the total original network size.
[Physical Significance]: Serving as the Order Parameter in Percolation Theory, it directly reflects whether the 'physical skeleton' 
          of the metro system maintains global-scale spatial reachability.
          It measures the proportion of the largest mutually connected cluster relative to the original scale.

[2. Global Transmission Efficiency (Global Efficiency, E)]
[Technical Definition]: The arithmetic mean of the reciprocal of the shortest path lengths between all node pairs in the network.
[Physical Significance]: Quantifies the 'Service Efficacy' of the system. Even if the physical structure has not fully fragmented, 
          the failure of core hubs can ca

interactive(children=(Dropdown(description='Target City:', options=('Beijing', 'Changchun', 'Changsha', 'Chang…